In [1]:
from processbehavior import ProcessBehavior
from processbehavior.datasets import synthetic

df = synthetic.make_sds(1, seed=42)

pb = ProcessBehavior(df)
print(pb.data.head(25))

%config Completer.use_jedi = False

    time factor 1 factor 2          y
0      1     F1_1     F2_1  51.346824
1      1     F1_1     F2_1  51.516211
2      2     F1_1     F2_1  52.666367
3      2     F1_1     F2_1  52.699215
4      2     F1_1     F2_1  52.242386
5      2     F1_1     F2_1  52.196936
6      2     F1_1     F2_1  52.725067
7      3     F1_1     F2_1  50.764118
8      3     F1_1     F2_1  50.821078
9      3     F1_1     F2_1  50.906611
10     4     F1_1     F2_1  52.386720
11     4     F1_1     F2_1  52.244826
12     4     F1_1     F2_1  52.464050
13     5     F1_1     F2_1  50.751212
14     5     F1_1     F2_1  50.938032
15     6     F1_1     F2_1  53.215166
16     6     F1_1     F2_1  53.521614
17     6     F1_1     F2_1  53.074134
18     6     F1_1     F2_1  53.193683
19     7     F1_1     F2_1  52.294774
20     7     F1_1     F2_1  52.967899
21     8     F1_1     F2_1  51.806120
22     8     F1_1     F2_1  51.998878
23     1     F1_1     F2_2  46.218850
24     1     F1_1     F2_2  46.306391


In [2]:
study = pb.formulate(
    response=pb.cols.y,        # Required: measurement variable
    time=pb.cols.time,         # Optional: time/sequence variable
    precision=3,
    factors=[pb.cols.factor_1,pb.cols.factor_2,pb.cols.time]
   # plan={'factors': {pb.cols.factor_1: ['K1', 'K2', 'K3' ]},'T':9,'N':4}
)
print(study.design())
print(study.sds)
result = study.execute()


More than 2 factors in RSG - calculating interaction for first 2: ['factor 1', 'factor 2']


DesignReport(3 factors, observed only)
  SDS reason: full_replication (min n = 2 >= 2)
  K: observed=48
  T: observed=8
  R: observed=48
  N: observed=(min=2, median=3.0, max=5)

  Factors:
    factor 1: observed=['F1_1', 'F1_2', 'F1_3']
    factor 2: observed=['F2_1', 'F2_2']
    time: observed=[1, 2, 3, 4, 5, 6, 7, 8]

  Structure: Complete structure
1


In [22]:
# Extract Xbar and S statistics using the new API
xbar_data = result.get_chart('Xbar')[['rsg', 'xbar', 'lpl', 'upl']].copy()
sbar_data = result.get_chart('S')[['rsg', 's']].copy()

# Combine into summary table
summary_stats = xbar_data.merge(sbar_data, on='rsg')

# Add observation counts
counts = result.dataset.groupby('rsg', observed=True).size().reset_index(name='n')
summary_stats = summary_stats.merge(counts, on='rsg')

# Sort by natural order
summary_stats = summary_stats.sort_values('rsg')

# Rename columns for display
summary_stats = summary_stats.rename(columns={'lpl': 'LPL', 'upl': 'UPL'})

print("Lane x Phase Xbar/S Statistics:")
print("=" * 70)
print(summary_stats.to_string(index=False))
print("=" * 70)
print(f"Total: {summary_stats['n'].sum()} observations across {len(summary_stats)} combinations")

# Check for signals
signals = result.detect_signals(chart='Xbar')
print(f"\nXbar Signals Detected: {signals.count}")
signals_s = result.detect_signals(chart='S')
print(f"Sbar Signals Detected: {signals_s.count}")

Lane x Phase Xbar/S Statistics:
        rsg   xbar    LPL    UPL     s  n
F1_1_F2_1_1 51.432 48.241 49.991 0.120  2
F1_1_F2_1_2 52.506 48.646 49.586 0.263  5
F1_1_F2_1_3 50.831 48.473 49.759 0.072  3
F1_1_F2_1_4 52.365 48.473 49.759 0.111  3
F1_1_F2_1_5 50.845 48.241 49.991 0.132  2
F1_1_F2_1_6 53.251 48.580 49.652 0.191  4
F1_1_F2_1_7 52.631 48.241 49.991 0.476  2
F1_1_F2_1_8 51.902 48.241 49.991 0.136  2
F1_1_F2_2_1 46.081 48.646 49.586 0.177  5
F1_1_F2_2_2 47.795 48.241 49.991 0.068  2
F1_1_F2_2_3 47.089 48.473 49.759 0.313  3
F1_1_F2_2_4 47.636 48.241 49.991 0.474  2
F1_1_F2_2_5 46.600 48.473 49.759 0.335  3
F1_1_F2_2_6 48.689 48.646 49.586 0.414  5
F1_1_F2_2_7 48.462 48.646 49.586 0.265  5
F1_1_F2_2_8 47.991 48.580 49.652 0.441  4
F1_2_F2_1_1 48.583 48.646 49.586 0.226  5
F1_2_F2_1_2 50.162 48.473 49.759 0.110  3
F1_2_F2_1_3 48.535 48.646 49.586 0.356  5
F1_2_F2_1_4 49.664 48.580 49.652 0.467  4
F1_2_F2_1_5 47.690 48.473 49.759 0.480  3
F1_2_F2_1_6 50.445 48.646 49.586 0.296  5
F1

In [3]:

print(result.get_statistics('Xbar'))

result.plot(chart='Xbar', theme='ggplot').show()

{'center': np.float64(49.116), 'N': 'Varies', 'lpl': 'Varies', 'upl': 'Varies'}


In [4]:
result.plot(chart='S').show()
print(result.get_statistics('S'))

{'center': np.float64(0.329), 'N': 'Varies', 'lpl': 'Varies', 'upl': 'Varies'}


In [10]:
study_no_t = pb.formulate(
    response=pb.cols.y,        # Required: measurement variable
    time=pb.cols.time,         # Optional: time/sequence variable
    precision=3,
    factors=[pb.cols.factor_1,pb.cols.factor_2]
   # plan={'factors': {pb.cols.factor_1: ['K1', 'K2', 'K3' ]},'T':9,'N':4}
)
print(study_no_t.design())
print(study_no_t.sds)
print(study_no_t.valid_charts)
print(study_no_t.residual_charts)

DesignReport(2 factors, observed only)
  SDS reason: full_replication (min n = 2 >= 2)
  K: observed=6
  T: observed=8
  R: observed=48
  N: observed=(min=2, median=3.0, max=5)

  Factors:
    factor 1: observed=['F1_1', 'F1_2', 'F1_3']
    factor 2: observed=['F2_1', 'F2_2']

  Structure: Complete structure
1
['Xbar', 'S', 'R', 'Imr']
['R2_S', 'R3_Xbar', 'R3_S', 'R4_Xbar', 'R4_S', 'R5_Xbar', 'R5_S']


In [8]:
result = study_no_t.execute(chart='R5_Xbar')
result.plot(chart='R5_Xbar', theme='ggplot').show()

In [15]:
print(result.summary)
result = study_no_t.execute(chart='R2_S')
result.plot(chart='R2_S', theme='ggplot').show()

{'sds': 1, 'sds_description': 'Full replication (all cells n≥2)', 'sds_capabilities': ['full_vas', 'all_residuals', 'interactions', 'main_effects'], 'replication_type': 'full', 'analysis_type': 'Xbar', 'response_var': 'y', 'grouping_vars': ['factor 1', 'factor 2'], 'time_var': 'time', 'n_observations': 161, 'n_charts': 2, 'chart_types': ['Xbar', 'S'], 'has_residuals': True, 'has_effects': True, 'has_interactions': True, 'variance_decomposition': True, 'interaction_analysis': True, 'n_signals_total': 5, 'is_stratified': False}
